# Etapa 5l — Familias nuevas de mapas para los vasos finos

**Ronda de mejoras para la entrega final (desvío 14.11 del plan).** Solo desarrollo, la misma validación cruzada por personas. El conjunto de prueba no se usa.

**Motivo.** La limitación principal del algoritmo son los vasos finos: encuentra cerca del 35 % de su línea central. Agregar mapas del mismo tipo ya rinde milésimas (textura, borde asimétrico, mapas contra lesiones), y los otros canales de color no suman (`02d`). Se prueban cuatro familias de mapas que no tenemos, respaldadas por artículos, ordenadas por cuánto atacan el problema de los finos:

| Familia | Artículo | Qué aporta | Mapas |
|---|---|---|---|
| **Simetría de fase** (`enhance.phase_symmetry`) | Kovesi (1999); Läthén, Jonasson y Borga (2010) | Nuestros filtros responden en proporción al contraste; la fase no: un vaso fino tenue da casi lo mismo que uno marcado. Filtros log-Gabor en cuadratura (8 orientaciones, longitud de onda 2 × ancho de cada escala de la etapa 4). | 6: todas las escalas y cada una |
| **B-COSFIRE** (`enhance.bcosfire`) | Azzopardi, Strisciuglio, Vento y Petkov (2015) | Media geométrica de respuestas de diferencia de gaussianas a lo largo de una línea: responde a barras más que a manchas; la versión asimétrica, también a las terminaciones de los vasos. Las dos configuraciones del artículo para DRIVE, llevadas al diámetro del campo, y la simétrica a un tercio de esa escala (la base resuelve vasos unas tres veces más finos que DRIVE). | 3 |
| **Aperturas por caminos** (`enhance.path_openings`) | Talbot y Appleton (2007) | Conservan lo claro que contiene un camino largo que puede curvarse (hasta 45° de una dirección), en lugar de un segmento recto: no pierden los vasos finos tortuosos. Largos de 2 % y 4 % del diámetro, los de la apertura por reconstrucción. | 2 |
| **Autovalores del hessiano** (`enhance.hessian_eigenvalues`) | Niemeijer et al. (2004); Staal et al. (2004) | Frangi resume los dos autovalores en un número con parámetros fijos; sueltos, la regresión aprende la combinación. En las 5 escalas de la etapa 4. | 10 |

Las pruebas de cada familia están en `tests/test_enhance.py` (la de los caminos se compara con una búsqueda directa). Los parámetros son los de los artículos o de criterio; no se ajustaron con datos.

**Candidatas** (sobre el algoritmo actual: Wiener con ruido estimado, relleno de exudados, división, entrenamiento equilibrado por grosor, valor fijo elegido en entrenamiento):
- **A.** Los 28 mapas actuales.
- **B.** A + simetría de fase (34). **C.** A + B-COSFIRE (31). **D.** A + aperturas por caminos (30). **E.** A + autovalores del hessiano (38).
- **F.** A + las cuatro familias (49).

**Medidas**, en las personas de afuera, a resolución original, en el campo de visión de la base: vasos finos (sensibilidad de línea central), Matthews, Dice de línea central y precisión. Descriptivo: por grupo.

**Regla de adopción (fijada antes de correr).** Como estas familias apuntan a los vasos finos, una candidata **cumple** si
1. mejora los vasos finos con un intervalo al 95 % que no incluye el cero, **y**
2. no es inferior a A en Matthews ni en Dice de línea central (límite inferior > −0,03).

Si varias cumplen, gana la de menos mapas entre las que no son inferiores en vasos finos a la que más los mejora (margen −0,03). Si ninguna cumple, queda A. Después de esta ronda viene la selección con L1 sobre los mapas que queden.

**Control:** A tiene que reproducir el Matthews de B en `05k` (0,7525).

In [ ]:
import sys
import time
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd

from src import config as C
from src.data import dev_ids, load_split
from src.evaluation import map_images
from src.pipeline import choose_value, fit_weights
from src.split import person_folds
from src.stats import percentile_interval, person_bootstrap_means

pd.set_option("display.width", 220)
ids = dev_ids()
dev = load_split().set_index("image").loc[ids]
folds = dev["person"].map(person_folds(dev.reset_index(), 5, C.SEED))
MODE = C.SEGMENT_BACKGROUND_MODE
FAM = C.NEW_FEATURE_FAMILIES
SUPER = C.SEGMENT_FEATURES + [n for names in FAM.values() for n in names]
CANDS = {"A. actual": C.SEGMENT_FEATURES,
         "B. + fase": C.SEGMENT_FEATURES + FAM["fase"],
         "C. + B-COSFIRE": C.SEGMENT_FEATURES + FAM["B-COSFIRE"],
         "D. + caminos": C.SEGMENT_FEATURES + FAM["caminos"],
         "E. + hessiano": C.SEGMENT_FEATURES + FAM["hessiano"],
         "F. + todas": SUPER}
COLS = {c: [SUPER.index(n) for n in names] for c, names in CANDS.items()}
ORDER = list(CANDS)
WORKERS = 4
print(MODE, "|", {c: len(v) for c, v in CANDS.items()})

## Pasada 1: píxeles de entrenamiento con los 49 mapas (cada candidata usa sus columnas)

In [ ]:
def pass1(image_id):
    from src.pipeline import training_pixels
    return image_id, training_pixels(image_id, background=MODE, features=SUPER)

t = time.time()
samples_all = dict(map_images(pass1, ids, workers=WORKERS))
print(f"Pasada 1: {time.time() - t:.0f} s")


def subset(c):
    keys_x = ("x", "x_thr")
    return {i: {k: (v[:, COLS[c]] if k in keys_x else v) for k, v in s.items()} for i, s in samples_all.items()}

## Ajuste por vuelta

In [ ]:
fit = {}
for c in CANDS:
    sc = subset(c)
    for k in range(5):
        tr = list(folds.index[folds != k])
        w = fit_weights(sc, tr)
        full = np.zeros(1 + len(SUPER))
        full[0] = w[0]
        full[1 + np.array(COLS[c])] = w[1:]
        fit[(k, c)] = (full, choose_value(sc, w, tr))
pd.DataFrame({c: [fit[(k, c)][1] for k in range(5)] for c in CANDS}, index=[f"vuelta {k}" for k in range(5)]).round(4)

## Pasada 2: las personas de afuera (los mapas se calculan una vez para las 6 candidatas)

In [ ]:
def pass2(image_id):
    import numpy as np
    from src.config import WIDTH_LIMITS_PX
    from src.data import load_fov, load_image, load_vessels
    from src.enhance import upscale
    from src.fov import detect_fov_robust
    from src.metrics import centerline_dice, matthews
    from src.pipeline import feature_scores
    from src.vessels import skeleton_widths
    k = int(folds[image_id])
    rgb = load_image(image_id)
    truth, fov = load_vessels(image_id), load_fov(image_id)
    auto = detect_fov_robust(rgb)
    skel, wd = skeleton_widths(truth & fov)
    thin = skel & (wd <= WIDTH_LIMITS_PX[0])
    probs, _, _ = feature_scores(rgb, [fit[(k, c)][0] for c in ORDER], 2, background=MODE, features=SUPER)
    rows = []
    for c, p in zip(ORDER, probs):
        m = (upscale(p, fov.shape) > fit[(k, c)][1]) & auto
        marked = m & fov
        rows.append({"image": image_id, "candidata": c, "finos": m[thin].mean(), "matthews": matthews(m, truth, fov),
                     "dice_linea_central": centerline_dice(m, truth, fov),
                     "precision": (marked & truth).sum() / max(marked.sum(), 1)})
    return rows

t = time.time()
R = pd.DataFrame([r for rows in map_images(pass2, ids, workers=WORKERS) for r in rows])
R.to_csv(ROOT / "results" / "05l_mapas_nuevos.csv", index=False)
print(f"Pasada 2: {time.time() - t:.0f} s")

## Resultados

In [ ]:
MEAS = ("finos", "matthews", "dice_linea_central", "precision")


def diff(new, old, meas):
    w = R.pivot(index="image", columns="candidata", values=meas)
    d = (w[new] - w[old]).to_frame()
    lo, hi = percentile_interval(person_bootstrap_means(d, dev["person"]))
    return d.iloc[:, 0].mean(), lo[0], hi[0], int((d.iloc[:, 0] > 0).sum())


rows = []
for c in ORDER[1:]:
    for meas in MEAS:
        m, lo, hi, n = diff(c, ORDER[0], meas)
        rows.append({"candidata": c, "medida": meas, "diferencia con A": m, "inferior": lo, "superior": hi,
                     "imágenes que mejoran": n})
table = pd.DataFrame(rows).set_index(["candidata", "medida"])
print("Medias:")
display(R.pivot_table(index="candidata", values=list(MEAS)).loc[ORDER, list(MEAS)].round(4))
table.round(4)

In [ ]:
meets = [c for c in ORDER[1:]
         if table.loc[(c, "finos"), "inferior"] > 0
         and table.loc[(c, "matthews"), "inferior"] > -C.NONINF_MARGIN_MATTHEWS
         and table.loc[(c, "dice_linea_central"), "inferior"] > -C.NONINF_MARGIN_MATTHEWS]
print("Cumplen:", meets)
if meets:
    best = max(meets, key=lambda c: table.loc[(c, "finos"), "diferencia con A"])
    close = [c for c in meets if c == best or diff(c, best, "finos")[1] > -C.NONINF_MARGIN_MATTHEWS]
    adopt = min(close, key=lambda c: (len(CANDS[c]), ORDER.index(c)))
    print("La que más mejora los finos:", best, "| no inferiores a ella:", close)
else:
    adopt = ORDER[0]
print("→ se adopta:", adopt, f"({len(CANDS[adopt])} mapas)")

In [ ]:
print("Por grupo (media):")
R.join(dev["group"], on="image").pivot_table(index="candidata", columns="group",
                                              values=["finos", "matthews", "precision"]).loc[ORDER].round(3)

## Decisión (escrita después de correr)

*(pendiente)*